# 02b · Generación para la evaluación, sin relleno: base frente a adaptadores (Colab T4)

**TraceGit · ISIA-119 · Grupo 3.** Genera la respuesta de cada condición (modelo base sin adaptador, adaptador v1
y, más adelante, v2) para los 375 ejemplos del **test de la v1.1** del dataset. Este notebook **solo genera**: las
métricas (ROUGE-L, BLEU, exactitud de identificadores, idioma correcto) y el análisis de sesgo D1 se calculan en
local con `eval/metricas_especializacion.py`, que no necesita GPU.

- Decodificación **voraz** (`do_sample=False`), máximo 128 tokens nuevos, mismo prompt de sistema que en el
  entrenamiento y `enable_thinking=False`. Todas las condiciones usan la misma configuración.
- Las predicciones se suben a `brenpc/tracegit-predicciones` (dataset privado) cada 10 lotes. Si Colab se
  corta: reconectar y **Run All**; se retoma donde quedó.
- Para agregar el adaptador v2: añadir su línea en `ADAPTADORES` (celda 1) y volver a ejecutar. Las condiciones
  ya terminadas no se repiten.

**Por qué existe este notebook (02b).** En la corrida del `02` (carpeta `test_v1.1` de las predicciones), todas las filas de un lote que llevaban **relleno a la izquierda** salieron vacías: 316 de 375, en base y en v1 por igual. Solo funcionaron las filas sin relleno (375/375 coinciden). Gemma 4 E2B no genera bien con relleno a la izquierda en este entorno. Aquí cada lote junta solo prompts con **el mismo número de tokens**, así que no hay relleno y el resultado es el mismo que generar de a uno. Las predicciones van a una carpeta nueva (`test_v1.1_r2`); la del `02` se conserva como evidencia del fallo. Un control detiene la corrida si aparecen predicciones vacías.

**Cómo ejecutarlo:** VS Code → *Select Kernel* → Colab → **GPU T4** → *Run All*. La celda 2 pide el token de
Hugging Face (permiso de **escritura**) en un cuadro de texto: no pegarlo en el notebook ni en el chat.

## 1. Configuración

In [ ]:
MODELO = dict(repo="google/gemma-4-E2B-it", revision="4adf15748de9e1a0097b6274cee9c15836c41c37")
DATASET = dict(repo="brenpc/tracegit-commits", revision="42eed230d333d04b7ad196032a6fe1d88524e323",
               archivo="v1.1/test.jsonl")
ADAPTADORES = {
    "v1": dict(repo="brenpc/tracegit-gemma4-e2b-lora-v1", revision="9157fe0c5e22b8ce6a02723d2f6e1c9c23694636"),
    # "v2": dict(repo="brenpc/tracegit-gemma4-e2b-lora-v2", revision="<SHA>"),
}
CONDICIONES = ["base"] + list(ADAPTADORES)

REPO_PRED = "brenpc/tracegit-predicciones"      # dataset privado con las predicciones
ETIQUETA = "test_v1.1_r2"                       # carpeta dentro de REPO_PRED (test_v1.1 = corrida fallida del 02)
SALIDA = "/content/predicciones"

MAX_NUEVOS = 128
LOTE = 8                                         # máximo por lote; solo prompts del mismo largo
MAX_VACIAS = 0.05                                # se detiene si más del 5 % sale vacío
SUBIR_CADA = 10                                  # lotes
N_LIMITE = None                                  # p. ej. 16 para una prueba rápida

## 2. Dependencias y token de Hugging Face

Mismas versiones que el entrenamiento (`01_entrenamiento_qlora.ipynb`).

In [ ]:
%pip install -q "transformers==5.23.0" "peft==0.21.3" "bitsandbytes==0.50.2" "accelerate==1.16.0"

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Token de Hugging Face (permiso de escritura): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Sesión de Hugging Face:", whoami()["name"])

## 3. Entorno y GPU libre

Si otro kernel del mismo servidor (por ejemplo el del entrenamiento) retiene la GPU, se cierra.

In [ ]:
import json, platform, signal, subprocess, time
import torch, transformers, peft, bitsandbytes, accelerate

assert torch.cuda.is_available(), "No hay GPU: conectar el kernel a un servidor Colab con T4."

def liberar_gpu():
    yo = os.getpid()
    sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
    pids = {int(l.split()[0]) for l in sh("ps -eo pid,cmd | grep -E 'ipykernel|colab_kernel_launcher' | grep -v grep").splitlines()}
    pids |= {int(p) for p in sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader").split() if p.isdigit()}
    for pid in sorted(pids - {yo, os.getppid()}):
        try:
            print("  cerrando el proceso", pid); os.kill(pid, signal.SIGKILL)
        except (ProcessLookupError, PermissionError) as e:
            print("  no se pudo cerrar", pid, type(e).__name__)
    time.sleep(5)

if torch.cuda.mem_get_info()[0] / 2**30 < 11:
    print("Hay otro kernel usando la GPU. Intentando liberarla...")
    liberar_gpu()
libre, total = (x / 2**30 for x in torch.cuda.mem_get_info())
assert libre >= 11, f"Solo {libre:.1f} GB libres: finalizar las otras sesiones de Colab y conectar una T4 nueva."

ENTORNO = {"gpu": torch.cuda.get_device_name(0), "vram_total_gb": round(total, 2),
           "python": platform.python_version(), "torch": torch.__version__,
           "transformers": transformers.__version__, "peft": peft.__version__,
           "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__}
print(json.dumps(ENTORNO, indent=2))

## 4. Test de la v1.1 (revisión fijada)

In [ ]:
from collections import Counter
from huggingface_hub import hf_hub_download

ruta = hf_hub_download(DATASET["repo"], DATASET["archivo"], repo_type="dataset", revision=DATASET["revision"])
test = [json.loads(l) for l in open(ruta, encoding="utf-8")]
if N_LIMITE:
    test = test[:N_LIMITE]
print(len(test), "ejemplos ·", dict(Counter(x["meta"]["idioma"] for x in test)),
      "·", dict(Counter(x["meta"]["estilo_repo"] for x in test)))

## 5. Modelo en 4 bits y adaptadores

Misma cuantización que el entrenamiento (NF4, cómputo fp16). Los adaptadores se cargan sobre el mismo modelo; la condición `base` desactiva el adaptador con `disable_adapter()`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODELO["repo"], revision=MODELO["revision"])
if tok.chat_template is None:
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODELO["repo"], revision=MODELO["revision"]).chat_template
tok.padding_side = "left"
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
FIN = [i for i in {tok.eos_token_id, tok.convert_tokens_to_ids("<turn|>")} if i is not None and i >= 0]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16,
                         llm_int8_skip_modules=["vision_tower", "audio_tower", "embed_vision", "embed_audio", "lm_head"])
base = AutoModelForMultimodalLM.from_pretrained(MODELO["repo"], revision=MODELO["revision"],
                                                quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
nombres = list(ADAPTADORES)
model = PeftModel.from_pretrained(base, ADAPTADORES[nombres[0]]["repo"], revision=ADAPTADORES[nombres[0]]["revision"],
                                  adapter_name=nombres[0])
for n in nombres[1:]:
    model.load_adapter(ADAPTADORES[n]["repo"], revision=ADAPTADORES[n]["revision"], adapter_name=n)
model.eval(); model.config.use_cache = True
print("adaptadores:", list(model.peft_config), "· VRAM", round(torch.cuda.memory_allocated() / 2**30, 2), "GB")

PROMPTS = {x["meta"]["sha"]: tok.apply_chat_template(x["messages"][:-1], tokenize=False, add_generation_prompt=True,
                                                     enable_thinking=False) for x in test}
LARGO = {s: len(tok(p, add_special_tokens=False)["input_ids"]) for s, p in PROMPTS.items()}
print("largos de prompt distintos:", len(set(LARGO.values())), "· máx", max(LARGO.values()))

## 6. Generación con retoma

Por cada condición, `preds_<condición>.jsonl` guarda una línea por ejemplo. Al empezar se descarga lo que ya esté en el Hub y solo se generan los que faltan. Los lotes agrupan prompts con exactamente el mismo número de tokens: **sin relleno**.

In [ ]:
from contextlib import nullcontext
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(REPO_PRED, repo_type="dataset", private=True, exist_ok=True)
os.makedirs(SALIDA, exist_ok=True)
archivos_hub = set(api.list_repo_files(REPO_PRED, repo_type="dataset"))

def ruta_local(cond):
    return os.path.join(SALIDA, f"preds_{cond}.jsonl")

def subir(cond):
    api.upload_file(path_or_fileobj=ruta_local(cond), path_in_repo=f"{ETIQUETA}/preds_{cond}.jsonl",
                    repo_id=REPO_PRED, repo_type="dataset", commit_message=f"{ETIQUETA}: {cond}")

def generar_lote(lote, cond):
    assert len({LARGO[x["meta"]["sha"]] for x in lote}) == 1, "lote con largos distintos: habría relleno"
    enc = tok([PROMPTS[x["meta"]["sha"]] for x in lote], return_tensors="pt", padding=False,
              add_special_tokens=False).to(model.device)
    enc.pop("token_type_ids", None)
    if cond != "base":
        model.set_adapter(cond)
    with torch.no_grad(), (model.disable_adapter() if cond == "base" else nullcontext()):
        out = model.generate(**enc, max_new_tokens=MAX_NUEVOS, do_sample=False, use_cache=True,
                             eos_token_id=FIN, pad_token_id=PAD)
    nuevos = out[:, enc["input_ids"].shape[1]:]
    res = []
    for x, fila in zip(lote, nuevos):
        ids = fila.tolist()
        n = next((i for i, t in enumerate(ids) if t in FIN), len(ids))
        res.append({"sha": x["meta"]["sha"], "repo": x["meta"]["repo"], "condicion": cond,
                    "prediccion": tok.decode(ids[:n], skip_special_tokens=True).strip(),
                    "tokens": n, "truncada": n >= MAX_NUEVOS})
    return res

TIEMPOS = {}
for cond in CONDICIONES:
    hecho = {}
    if f"{ETIQUETA}/preds_{cond}.jsonl" in archivos_hub:
        p = hf_hub_download(REPO_PRED, f"{ETIQUETA}/preds_{cond}.jsonl", repo_type="dataset", force_download=True)
        hecho = {r["sha"]: r for r in map(json.loads, open(p, encoding="utf-8"))}
    with open(ruta_local(cond), "w", encoding="utf-8") as f:
        for r in hecho.values():
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    grupos = {}
    for x in test:
        if x["meta"]["sha"] not in hecho:
            grupos.setdefault(LARGO[x["meta"]["sha"]], []).append(x)
    pendientes = [g for _, g in sorted(grupos.items(), reverse=True)]
    n_faltan = sum(map(len, pendientes))
    print(f"[{cond}] hechos {len(hecho)} · faltan {n_faltan} en {len(pendientes)} grupos de igual largo")
    t0, lote_n, i, k, vacias = time.time(), LOTE, 0, 0, 0
    while pendientes:
        grupo = pendientes[0]
        lote = grupo[:lote_n]
        try:
            res = generar_lote(lote, cond)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache(); lote_n = max(1, lote_n // 2); print("  sin memoria: lote", lote_n); continue
        del grupo[:len(lote)]
        if not grupo:
            pendientes.pop(0)
        with open(ruta_local(cond), "a", encoding="utf-8") as f:
            for r in res:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        i += len(lote); k += 1
        vacias += sum(r["prediccion"] == "" for r in res)
        if i >= 40 and vacias / i > MAX_VACIAS:
            subir(cond)
            raise RuntimeError(f"[{cond}] {vacias} de {i} predicciones vacías: se detiene la corrida para revisar.")
        if k % SUBIR_CADA == 0:
            subir(cond); print(f"  {i}/{n_faltan} · {time.time() - t0:.0f}s · vacías {vacias} · subido")
    if n_faltan:
        subir(cond)
    TIEMPOS[cond] = round(time.time() - t0, 1)
    print(f"[{cond}] listo en {TIEMPOS[cond]:.0f}s")

## 7. Registro de la corrida y resumen rápido

Las métricas completas: en local, `python eval/metricas_especializacion.py`.

In [ ]:
REGISTRO = {"modelo_base": f"{MODELO['repo']}@{MODELO['revision']}",
            "dataset": f"{DATASET['repo']}@{DATASET['revision']}:{DATASET['archivo']}",
            "adaptadores": {k: f"{v['repo']}@{v['revision']}" for k, v in ADAPTADORES.items()},
            "decodificacion": {"do_sample": False, "max_new_tokens": MAX_NUEVOS, "enable_thinking": False,
                               "lotes": "solo prompts del mismo largo, sin relleno",
                               "cuantizacion": "nf4 + double quant, fp16"},
            "ejemplos": len(test), "segundos_por_condicion_ultima_sesion": TIEMPOS, "entorno": ENTORNO,
            "fecha": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())}
with open(os.path.join(SALIDA, "registro.json"), "w", encoding="utf-8") as f:
    json.dump(REGISTRO, f, ensure_ascii=False, indent=2)
api.upload_file(path_or_fileobj=os.path.join(SALIDA, "registro.json"), path_in_repo=f"{ETIQUETA}/registro.json",
                repo_id=REPO_PRED, repo_type="dataset", commit_message=f"{ETIQUETA}: registro")
print("Revisión de las predicciones:", api.dataset_info(REPO_PRED).sha)

preds = {c: {r["sha"]: r for r in map(json.loads, open(ruta_local(c), encoding="utf-8"))} for c in CONDICIONES}
for c in CONDICIONES:
    v = list(preds[c].values())
    print(f"{c:5s} n={len(v)} · tokens medios {sum(r['tokens'] for r in v)/len(v):.1f} · "
          f"truncadas {sum(r['truncada'] for r in v)} · vacías {sum(r['prediccion'] == '' for r in v)} · "
          f"'Add files via upload' "
          f"{sum(r['prediccion'].strip() == 'Add files via upload' for r in v)}")
import random
for x in random.Random(7).sample(test, 3):
    s = x["meta"]["sha"]
    print(f"\n[{x['meta']['idioma']} · {x['meta']['lenguaje']}] HUMANO: {x['messages'][-1]['content'][:120]!r}")
    for c in CONDICIONES:
        print(f"  {c:5s}: {preds[c][s]['prediccion'][:120]!r}")